# Bee-Wo Baseline Training on Google Colab

This notebook mounts Google Drive, copies the DL project files into Colab, unzips `data_clean.zip`, runs a smoke test, runs one selected baseline model end-to-end, and copies the results back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab config
from pathlib import Path

PROJECT_DIR_IN_DRIVE = '/content/drive/MyDrive/dl_project'
DATA_ZIP_IN_DRIVE = '/content/drive/MyDrive/dl_project/data_clean.zip'
RESULTS_DIR_IN_DRIVE = '/content/drive/MyDrive/dl_project/colab_runs'

MODEL_NAME = 'resnet10_3d'  # simple_cnn | resnet10_3d | r2plus1d_18 | mobilenetv2_3d | cnn_lstm | resnet10_temporal_attention | resnet10_landmark_fusion | mediapipe_logreg | mediapipe_random_forest
EPOCHS = 10
BATCH_SIZE = 4
IMAGE_SIZE = 64
SEQ_LEN = 16
SEED = 20260331

WORKDIR = Path('/content/dl_project')
WORKDIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = Path(RESULTS_DIR_IN_DRIVE)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print('Project dir in Drive:', PROJECT_DIR_IN_DRIVE)
print('Dataset zip in Drive:', DATA_ZIP_IN_DRIVE)
print('Model:', MODEL_NAME)

In [ ]:
import shutil
from pathlib import Path

drive_project = Path(PROJECT_DIR_IN_DRIVE)
assert drive_project.exists(), f'Missing project directory: {drive_project}'
assert Path(DATA_ZIP_IN_DRIVE).exists(), f'Missing dataset zip: {DATA_ZIP_IN_DRIVE}'

files_to_copy = [
    'train_baseline.py',
    'bee_wo_dataset.py',
    'data_clean/annotations.csv',
    'data_clean/splits.csv',
    'data_clean/label_map.csv',
    'baseline_models/__init__.py',
    'baseline_models/mediapipe_baseline.py',
    'baseline_models/resnet10_3d.py',
    'baseline_models/mobilenetv2_3d.py',
    'project_models/__init__.py',
    'project_models/r2plus1d_18.py',
    'project_models/cnn_lstm.py',
    'project_models/resnet10_temporal_attention.py',
    'project_models/resnet10_landmark_fusion.py',
    'train_mediapipe_baseline.py',
    'scripts/extract_mediapipe_landmarks.py',
]

for relative_path in files_to_copy:
    src = drive_project / relative_path
    dst = WORKDIR / relative_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

shutil.copy2(DATA_ZIP_IN_DRIVE, WORKDIR / 'data_clean.zip')
print('Copied project files into', WORKDIR)

In [ ]:
%cd /content/dl_project
!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet torch torchvision pillow scikit-learn numpy mediapipe

In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Selected device:', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path('/content/dl_project/data_clean.zip')
extract_root = Path('/content/dl_project')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_root)

all_dir = extract_root / 'data_clean' / 'all'
assert all_dir.exists(), f'Missing extracted dataset directory: {all_dir}'
print('Extracted data_clean.zip to', extract_root)
print('Gesture folders:', sorted(p.name for p in all_dir.iterdir() if p.is_dir()))

In [ ]:
!python - <<'PY'
from pathlib import Path
from bee_wo_dataset import BeeWoClipDataset

for split in ['train', 'val', 'test']:
    ds = BeeWoClipDataset(
        Path('data_clean'),
        Path('data_clean/annotations.csv'),
        Path('data_clean/splits.csv'),
        Path('data_clean/label_map.csv'),
        split=split,
        seq_len=16,
        image_size=64,
    )
    print(split, len(ds), tuple(ds[0]['frames'].shape))

## Landmark Extraction

This only runs for `resnet10_landmark_fusion`. Other models skip this step.

In [ ]:
import subprocess

if MODEL_NAME in {'resnet10_landmark_fusion', 'mediapipe_logreg', 'mediapipe_random_forest'}:
    cmd = [
        'python',
        'scripts/extract_mediapipe_landmarks.py',
        '--seq-len',
        str(SEQ_LEN),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f'Skipping landmark extraction for model: {MODEL_NAME}')


## Smoke Test

Run this once before the full training job. It verifies imports, data loading, batching, checkpoint writing, and metrics generation.

In [ ]:
import subprocess

if MODEL_NAME in {'mediapipe_logreg', 'mediapipe_random_forest'}:
    classifier_name = 'logreg' if MODEL_NAME == 'mediapipe_logreg' else 'random_forest'
    smoke_cmd = [
        'python',
        'train_mediapipe_baseline.py',
        '--classifier', classifier_name,
        '--seq-len', str(SEQ_LEN),
        '--landmarks-root', 'data_clean/landmarks',
        '--max-train-samples', '32',
    ]
else:
    smoke_cmd = [
        'python',
        'train_baseline.py',
        '--model', MODEL_NAME,
        '--epochs', '1',
        '--batch-size', str(BATCH_SIZE),
        '--image-size', str(IMAGE_SIZE),
        '--seq-len', str(SEQ_LEN),
        '--seed', str(SEED),
        '--max-train-batches', '1',
        '--max-eval-batches', '1',
    ]
    if MODEL_NAME == 'resnet10_landmark_fusion':
        smoke_cmd.extend(['--landmarks-root', 'data_clean/landmarks'])
print('Running:', ' '.join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)


## Real Training Run

In [ ]:
import subprocess

if MODEL_NAME in {'mediapipe_logreg', 'mediapipe_random_forest'}:
    classifier_name = 'logreg' if MODEL_NAME == 'mediapipe_logreg' else 'random_forest'
    train_cmd = [
        'python',
        'train_mediapipe_baseline.py',
        '--classifier', classifier_name,
        '--seq-len', str(SEQ_LEN),
        '--landmarks-root', 'data_clean/landmarks',
    ]
else:
    train_cmd = [
        'python',
        'train_baseline.py',
        '--model', MODEL_NAME,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--image-size', str(IMAGE_SIZE),
        '--seq-len', str(SEQ_LEN),
        '--seed', str(SEED),
    ]
    if MODEL_NAME == 'resnet10_landmark_fusion':
        train_cmd.extend(['--landmarks-root', 'data_clean/landmarks'])
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)


In [ ]:
import json
from pathlib import Path

runs_dir = Path('runs')
run_dirs = sorted([p for p in runs_dir.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
latest_run = run_dirs[-1]
print('Latest run:', latest_run)

metrics = json.loads((latest_run / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))

In [ ]:
from pathlib import Path

print('Validation confusion matrix:')
print((latest_run / 'val_confusion_matrix.csv').read_text())

print('Test confusion matrix:')
print((latest_run / 'test_confusion_matrix.csv').read_text())

In [ ]:
import shutil
destination = RESULTS_PATH / latest_run.name
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(latest_run, destination)
print('Copied run to Drive:', destination)